# Advanced ECG Classification Improvements
# MIT-BIH Arrhythmia Dataset

This notebook implements 6 major improvements:
1. Patient-wise Cross Validation
2. Replace SMOTE with Focal Loss + WeightedRandomSampler + MixUp
3. Temporal Convolutional Network (TCN)
4. Soft Voting Ensemble
5. SimCLR Contrastive Pretraining
6. Grad-CAM Visualization

In [ ]:
# Install dependencies
!pip install wfdb torch scikit-learn matplotlib seaborn numpy pandas tqdm

In [ ]:
# Import libraries
import wfdb
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from collections import Counter
import random
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Data Loading and Preprocessing
Load MIT-BIH Arrhythmia Dataset with patient IDs preserved

In [ ]:
# Download MIT-BIH Database
wfdb.dl_database('mitdb', dl_dir='mitdb', keep_subdirs=True)

In [ ]:
# AAMI classification mapping
AAMI_map = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    'V': 'V', 'E': 'V',
    'F': 'F'
}

class_to_int = {'N': 0, 'S': 1, 'V': 2, 'F': 3}
int_to_class = {0: 'N', 1: 'S', 2: 'V', 3: 'F'}

# All MIT-BIH records
all_records = [
    '100', '101', '103', '105', '106', '108', '109', '111', '112', '113',
    '114', '115', '116', '117', '118', '119', '121', '122', '123', '124',
    '200', '201', '202', '203', '205', '207', '208', '209', '210', '212',
    '213', '214', '215', '219', '220', '221', '222', '223', '228', '230',
    '231', '232', '233', '234'
]

print(f"Total records: {len(all_records)}")

In [ ]:
def extract_beats_with_patient_id(record_list, window_size=180):
    """
    Extract ECG beats with patient ID preserved
    
    Returns:
        beats: numpy array of ECG segments [N, window_size]
        labels: numpy array of class labels [N]
        patient_ids: numpy array of patient IDs [N]
    """
    beats = []
    labels = []
    patient_ids = []
    
    for rec in tqdm(record_list, desc="Extracting beats"):
        try:
            record = wfdb.rdrecord(f'mitdb/{rec}')
            annotation = wfdb.rdann(f'mitdb/{rec}', 'atr')
            
            signal = record.p_signal[:, 0]  # First channel
            r_peaks = annotation.sample
            symbols = annotation.symbol
            
            for i in range(len(r_peaks)):
                # Extract beat centered on R-peak
                start = r_peaks[i] - window_size // 2
                end = r_peaks[i] + window_size // 2
                
                if start >= 0 and end < len(signal):
                    beat = signal[start:end]
                    
                    # Map to AAMI class
                    symbol = symbols[i]
                    if symbol in AAMI_map:
                        aami_class = AAMI_map[symbol]
                        label = class_to_int[aami_class]
                        
                        beats.append(beat)
                        labels.append(label)
                        patient_ids.append(rec)  # Patient ID is the record number
        except Exception as e:
            print(f"Error processing record {rec}: {e}")
            continue
    
    beats = np.array(beats, dtype=np.float32)
    labels = np.array(labels, dtype=np.int64)
    patient_ids = np.array(patient_ids)
    
    return beats, labels, patient_ids

In [ ]:
# Extract all data with patient IDs
X_all, y_all, patient_ids = extract_beats_with_patient_id(all_records)

print(f"\nTotal beats extracted: {len(X_all)}")
print(f"Unique patients: {len(np.unique(patient_ids))}")
print(f"\nClass distribution:")
for i, class_name in int_to_class.items():
    count = np.sum(y_all == i)
    print(f"  {class_name}: {count} ({100*count/len(y_all):.2f}%)")

print(f"\nData shapes:")
print(f"  X_all: {X_all.shape}")
print(f"  y_all: {y_all.shape}")
print(f"  patient_ids: {patient_ids.shape}")

---
# 1. Patient-wise Cross Validation

Ensures no patient leakage between train and test sets using StratifiedGroupKFold.

In [ ]:
class ECGDataset(Dataset):
    """PyTorch Dataset for ECG beats"""
    def __init__(self, X, y, transform=None):
        self.X = torch.FloatTensor(X).unsqueeze(1)  # [N, 1, 180]
        self.y = torch.LongTensor(y)
        self.transform = transform
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        
        if self.transform:
            x = self.transform(x)
        
        return x, y

In [ ]:
def patient_wise_cross_validation(X, y, patient_ids, model_class, n_folds=5, epochs=30, batch_size=128):
    """
    Perform patient-wise k-fold cross validation
    
    Args:
        X: ECG beats [N, 180]
        y: labels [N]
        patient_ids: patient identifiers [N]
        model_class: Model class to instantiate
        n_folds: number of folds
        epochs: training epochs per fold
        batch_size: batch size
    
    Returns:
        results: dict with average metrics
    """
    # Use StratifiedGroupKFold to maintain class distribution while grouping by patient
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_results = {
        'accuracy': [],
        'f1_macro': [],
        'f1_per_class': {0: [], 1: [], 2: [], 3: []}
    }
    
    print("=" * 80)
    print(f"Starting {n_folds}-Fold Patient-wise Cross Validation")
    print("=" * 80)
    
    for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=patient_ids), 1):
        print(f"\n{'='*80}")
        print(f"Fold {fold}/{n_folds}")
        print(f"{'='*80}")
        
        # Split data
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        train_patients = patient_ids[train_idx]
        test_patients = patient_ids[test_idx]
        
        # Verify no patient leakage
        train_patient_set = set(train_patients)
        test_patient_set = set(test_patients)
        overlap = train_patient_set & test_patient_set
        assert len(overlap) == 0, f"Patient leakage detected: {overlap}"
        
        print(f"Train: {len(X_train)} beats from {len(train_patient_set)} patients")
        print(f"Test: {len(X_test)} beats from {len(test_patient_set)} patients")
        print(f"No patient overlap: ✓")
        
        # Create datasets
        train_dataset = ECGDataset(X_train, y_train)
        test_dataset = ECGDataset(X_test, y_test)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
        
        # Initialize model
        model = model_class(num_classes=4).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        
        # Mixed precision training
        scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
        
        # Training loop
        best_acc = 0.0
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                
                optimizer.zero_grad()
                
                if scaler:
                    with torch.cuda.amp.autocast():
                        outputs = model(batch_x)
                        loss = criterion(outputs, batch_y)
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(batch_x)
                    loss = criterion(outputs, batch_y)
                    loss.backward()
                    optimizer.step()
                
                train_loss += loss.item()
            
            scheduler.step()
            
            # Validation every 5 epochs
            if (epoch + 1) % 5 == 0:
                model.eval()
                all_preds = []
                all_targets = []
                
                with torch.no_grad():
                    for batch_x, batch_y in test_loader:
                        batch_x = batch_x.to(device)
                        outputs = model(batch_x)
                        preds = torch.argmax(outputs, dim=1).cpu().numpy()
                        all_preds.extend(preds)
                        all_targets.extend(batch_y.numpy())
                
                acc = accuracy_score(all_targets, all_preds)
                print(f"  Epoch [{epoch+1}/{epochs}] Loss: {train_loss/len(train_loader):.4f} Acc: {acc:.4f}")
                
                if acc > best_acc:
                    best_acc = acc
        
        # Final evaluation
        model.eval()
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(device)
                outputs = model(batch_x)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(batch_y.numpy())
        
        # Calculate metrics
        acc = accuracy_score(all_targets, all_preds)
        f1_macro = f1_score(all_targets, all_preds, average='macro')
        f1_per_class = f1_score(all_targets, all_preds, average=None)
        
        fold_results['accuracy'].append(acc)
        fold_results['f1_macro'].append(f1_macro)
        for i, f1 in enumerate(f1_per_class):
            fold_results['f1_per_class'][i].append(f1)
        
        print(f"\nFold {fold} Results:")
        print(f"  Accuracy: {acc:.4f}")
        print(f"  F1-macro: {f1_macro:.4f}")
        print(f"  Per-class F1:")
        for i, class_name in int_to_class.items():
            print(f"    {class_name}: {f1_per_class[i]:.4f}")
    
    # Calculate average results
    print(f"\n{'='*80}")
    print("Average Results Across All Folds")
    print(f"{'='*80}")
    
    avg_acc = np.mean(fold_results['accuracy'])
    avg_f1_macro = np.mean(fold_results['f1_macro'])
    
    print(f"Average Accuracy: {avg_acc:.4f} ± {np.std(fold_results['accuracy']):.4f}")
    print(f"Average F1-macro: {avg_f1_macro:.4f} ± {np.std(fold_results['f1_macro']):.4f}")
    print(f"\nAverage Per-class F1:")
    
    for i, class_name in int_to_class.items():
        avg_f1 = np.mean(fold_results['f1_per_class'][i])
        std_f1 = np.std(fold_results['f1_per_class'][i])
        print(f"  {class_name}: {avg_f1:.4f} ± {std_f1:.4f}")
    
    return fold_results

In [ ]:
# Simple CNN model for testing
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).squeeze(-1)
        x = self.fc(x)
        return x

In [ ]:
# Run patient-wise cross validation
# results = patient_wise_cross_validation(X_all, y_all, patient_ids, SimpleCNN, n_folds=5, epochs=20, batch_size=128)

---
# 2. Replace SMOTE with Focal Loss + WeightedRandomSampler + MixUp

Better handling of class imbalance without synthetic data generation.

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance
    Focuses training on hard-to-classify examples
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha  # Class weights [C]
        self.gamma = gamma  # Focusing parameter
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        """
        Args:
            inputs: [B, C] model outputs (logits)
            targets: [B] ground truth labels
        """
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)  # Probability of correct class
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
def create_weighted_sampler(labels, minority_oversample=3.0):
    """
    Create WeightedRandomSampler to balance training batches
    
    Args:
        labels: numpy array of labels
        minority_oversample: multiplier for minority class sampling
    
    Returns:
        WeightedRandomSampler instance
    """
    class_counts = np.bincount(labels)
    num_samples = len(labels)
    
    # Calculate weights inversely proportional to class frequency
    class_weights = num_samples / (len(class_counts) * class_counts)
    
    # Boost minority classes
    class_weights[1] *= minority_oversample  # S
    class_weights[3] *= minority_oversample  # F
    
    # Assign weight to each sample
    sample_weights = class_weights[labels]
    
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=num_samples,
        replacement=True
    )
    
    return sampler

In [ ]:
def mixup_data(x, y, alpha=0.2):
    """
    MixUp data augmentation for 1D ECG signals
    
    Args:
        x: input batch [B, C, L]
        y: labels [B]
        alpha: mixup interpolation strength
    
    Returns:
        mixed inputs, pairs of targets, and lambda
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """
    Mixed loss for MixUp
    """
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
class ECGDatasetWithTransform(Dataset):
    """Dataset with optional augmentation"""
    def __init__(self, X, y, use_mixup=False, mixup_alpha=0.2):
        self.X = torch.FloatTensor(X).unsqueeze(1)  # [N, 1, 180]
        self.y = torch.LongTensor(y)
        self.use_mixup = use_mixup
        self.mixup_alpha = mixup_alpha
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
def train_with_focal_weighted_mixup(model, train_loader, test_loader, epochs=50, device='cuda'):
    """
    Training loop with Focal Loss + WeightedRandomSampler + MixUp
    """
    # Class weights for focal loss (inverse frequency)
    # Adjust based on your dataset
    alpha = torch.tensor([1.0, 10.0, 5.0, 20.0], dtype=torch.float32).to(device)  # [N, S, V, F]
    criterion = FocalLoss(alpha=alpha, gamma=2.5)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    
    # Mixed precision
    scaler = torch.cuda.amp.GradScaler() if device == 'cuda' else None
    
    best_f1 = 0.0
    history = {'train_loss': [], 'val_acc': [], 'val_f1': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            # Apply MixUp with 50% probability
            if random.random() > 0.5:
                batch_x, y_a, y_b, lam = mixup_data(batch_x, batch_y, alpha=0.2)
                
                optimizer.zero_grad()
                
                if scaler:
                    with torch.cuda.amp.autocast():
                        outputs = model(batch_x)
                        loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
                    scaler.scale(loss).backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(batch_x)
                    loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
            else:
                optimizer.zero_grad()
                
                if scaler:
                    with torch.cuda.amp.autocast():
                        outputs = model(batch_x)
                        loss = criterion(outputs, batch_y)
                    scaler.scale(loss).backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(batch_x)
                    loss = criterion(outputs, batch_y)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
            
            train_loss += loss.item()
        
        scheduler.step()
        
        # Evaluation
        if (epoch + 1) % 5 == 0:
            model.eval()
            all_preds = []
            all_targets = []
            
            with torch.no_grad():
                for batch_x, batch_y in test_loader:
                    batch_x = batch_x.to(device)
                    outputs = model(batch_x)
                    preds = torch.argmax(outputs, dim=1).cpu().numpy()
                    all_preds.extend(preds)
                    all_targets.extend(batch_y.numpy())
            
            acc = accuracy_score(all_targets, all_preds)
            f1 = f1_score(all_targets, all_preds, average='macro')
            
            history['train_loss'].append(train_loss / len(train_loader))
            history['val_acc'].append(acc)
            history['val_f1'].append(f1)
            
            print(f"Epoch [{epoch+1}/{epochs}] Loss: {train_loss/len(train_loader):.4f} "
                  f"Acc: {acc:.4f} F1: {f1:.4f}")
            
            if f1 > best_f1:
                best_f1 = f1
                torch.save(model.state_dict(), 'best_focal_mixup_model.pth')
                print(f"  ✓ Saved best model (F1: {best_f1:.4f})")
    
    return history

In [ ]:
# Example: Setup DataLoader with WeightedRandomSampler
# Split data (use patient-wise split in practice)
from sklearn.model_selection import train_test_split

# For demonstration, simple split (use patient-wise in production)
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

# Create datasets
train_dataset = ECGDatasetWithTransform(X_train, y_train)
test_dataset = ECGDatasetWithTransform(X_test, y_test)

# Create weighted sampler
train_sampler = create_weighted_sampler(y_train, minority_oversample=3.0)

# DataLoaders
train_loader_weighted = DataLoader(train_dataset, batch_size=128, sampler=train_sampler, num_workers=0)
test_loader_weighted = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

print("DataLoaders ready with WeightedRandomSampler")
print(f"Train batches: {len(train_loader_weighted)}")
print(f"Test batches: {len(test_loader_weighted)}")

In [ ]:
# Train model with Focal Loss + Weighted Sampler + MixUp
# model_focal = SimpleCNN(num_classes=4).to(device)
# history = train_with_focal_weighted_mixup(model_focal, train_loader_weighted, test_loader_weighted, epochs=30, device=device)

---
# 3. Temporal Convolutional Network (TCN)

TCN with dilated causal convolutions for ECG classification.

In [ ]:
class TCNBlock(nn.Module):
    """
    Temporal Convolutional Network residual block
    with dilated causal convolutions
    """
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.3):
        super().__init__()
        
        # Calculate padding for causal convolution
        padding = (kernel_size - 1) * dilation
        
        self.conv1 = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation
        )
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.dropout1 = nn.Dropout(dropout)
        
        self.conv2 = nn.Conv1d(
            out_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation
        )
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(dropout)
        
        # Residual connection
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # Save input for residual
        residual = x if self.downsample is None else self.downsample(x)
        
        # First conv block
        out = self.conv1(x)
        out = out[:, :, :x.size(2)]  # Trim to original length (causal)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout1(out)
        
        # Second conv block
        out = self.conv2(out)
        out = out[:, :, :x.size(2)]  # Trim to original length (causal)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.dropout2(out)
        
        # Add residual
        out = out + residual
        out = self.relu(out)
        
        return out

In [ ]:
class TCN_ECG(nn.Module):
    """
    Temporal Convolutional Network for ECG beat classification
    
    Architecture:
    - Input: [B, 1, 180]
    - Multiple TCN blocks with increasing dilation (1, 2, 4, 8, 16)
    - Global average pooling
    - Classification head
    - Target: <2M parameters
    """
    def __init__(self, num_classes=4, num_channels=[32, 64, 128, 128, 128], 
                 kernel_size=3, dropout=0.3):
        super().__init__()
        
        # Initial projection
        self.input_conv = nn.Conv1d(1, num_channels[0], kernel_size=1)
        
        # TCN blocks with increasing dilation
        dilations = [1, 2, 4, 8, 16]
        self.tcn_blocks = nn.ModuleList()
        
        for i in range(len(dilations)):
            in_ch = num_channels[i] if i == 0 else num_channels[i-1]
            out_ch = num_channels[i]
            dilation = dilations[i]
            
            self.tcn_blocks.append(
                TCNBlock(in_ch, out_ch, kernel_size, dilation, dropout)
            )
        
        # Global pooling
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(num_channels[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        # Input projection
        x = self.input_conv(x)
        
        # Pass through TCN blocks
        for tcn_block in self.tcn_blocks:
            x = tcn_block(x)
        
        # Global pooling
        x = self.global_avg_pool(x).squeeze(-1)
        
        # Classification
        x = self.classifier(x)
        
        return x
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Create TCN model
tcn_model = TCN_ECG(num_classes=4, num_channels=[32, 64, 96, 96, 96], kernel_size=3, dropout=0.3).to(device)

print("TCN Model Architecture:")
print(tcn_model)
print(f"\nTotal parameters: {tcn_model.count_parameters():,}")
print(f"Target: <2M parameters {'✓' if tcn_model.count_parameters() < 2_000_000 else '✗'}")

# Test forward pass
dummy_input = torch.randn(2, 1, 180).to(device)
output = tcn_model(dummy_input)
print(f"\nTest output shape: {output.shape}")

In [ ]:
# Train TCN model
# history_tcn = train_with_focal_weighted_mixup(tcn_model, train_loader_weighted, test_loader_weighted, epochs=30, device=device)

---
# 4. Soft Voting Ensemble

Combine predictions from multiple trained models.

In [ ]:
class SoftVotingEnsemble:
    """
    Soft Voting Ensemble for multiple PyTorch models
    Averages softmax probabilities from all models
    """
    def __init__(self, models, device='cuda'):
        """
        Args:
            models: list of PyTorch models
            device: device to run inference
        """
        self.models = models
        self.device = device
        
        # Set all models to eval mode
        for model in self.models:
            model.eval()
            model.to(device)
    
    def predict_proba(self, x):
        """
        Get averaged softmax probabilities
        
        Args:
            x: input tensor [B, C, L]
        
        Returns:
            averaged probabilities [B, num_classes]
        """
        x = x.to(self.device)
        all_probs = []
        
        with torch.no_grad():
            for model in self.models:
                logits = model(x)
                probs = F.softmax(logits, dim=1)
                all_probs.append(probs)
        
        # Average probabilities
        avg_probs = torch.stack(all_probs).mean(dim=0)
        return avg_probs
    
    def predict(self, x):
        """
        Get final class predictions
        
        Args:
            x: input tensor [B, C, L]
        
        Returns:
            class predictions [B]
        """
        probs = self.predict_proba(x)
        return torch.argmax(probs, dim=1)
    
    def evaluate(self, data_loader):
        """
        Evaluate ensemble on a dataset
        
        Args:
            data_loader: PyTorch DataLoader
        
        Returns:
            metrics dict
        """
        all_preds = []
        all_targets = []
        
        for batch_x, batch_y in tqdm(data_loader, desc="Evaluating ensemble"):
            preds = self.predict(batch_x).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(batch_y.numpy())
        
        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)
        
        # Calculate metrics
        acc = accuracy_score(all_targets, all_preds)
        f1_macro = f1_score(all_targets, all_preds, average='macro')
        f1_per_class = f1_score(all_targets, all_preds, average=None)
        
        # Classification report
        print("\n" + "="*80)
        print("Ensemble Performance")
        print("="*80)
        print(f"Accuracy: {acc:.4f}")
        print(f"F1-macro: {f1_macro:.4f}")
        print(f"\nPer-class F1:")
        for i, f1 in enumerate(f1_per_class):
            print(f"  Class {int_to_class[i]}: {f1:.4f}")
        
        print("\n" + classification_report(
            all_targets, all_preds,
            target_names=['N', 'S', 'V', 'F'],
            digits=4
        ))
        
        # Confusion matrix
        cm = confusion_matrix(all_targets, all_preds)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['N', 'S', 'V', 'F'],
                    yticklabels=['N', 'S', 'V', 'F'])
        plt.title('Ensemble Confusion Matrix', fontsize=14, fontweight='bold')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.tight_layout()
        plt.show()
        
        return {
            'accuracy': acc,
            'f1_macro': f1_macro,
            'f1_per_class': f1_per_class,
            'predictions': all_preds,
            'targets': all_targets
        }

In [ ]:
def load_trained_models(model_configs, checkpoint_paths, device='cuda'):
    """
    Load multiple trained models from checkpoints
    
    Args:
        model_configs: list of (model_class, kwargs) tuples
        checkpoint_paths: list of checkpoint file paths
        device: device to load models
    
    Returns:
        list of loaded models
    """
    models = []
    
    for (model_class, kwargs), checkpoint_path in zip(model_configs, checkpoint_paths):
        model = model_class(**kwargs).to(device)
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        model.eval()
        models.append(model)
        print(f"Loaded {model_class.__name__} from {checkpoint_path}")
    
    return models

In [ ]:
# Example: Create ensemble from pre-trained models
# Assuming you have these models already trained and saved

# Model configurations
# model_configs = [
#     (ECGResNet, {'num_classes': 4}),
#     (AdvancedECGClassifier, {'num_classes': 4}),
#     (ImprovedECGClassifier, {'num_classes': 4})
# ]

# checkpoint_paths = [
#     'best_model_resnet.pth',
#     'best_model_advanced.pth',
#     'best_model_improved.pth'
# ]

# Load models
# models = load_trained_models(model_configs, checkpoint_paths, device=device)

# Create ensemble
# ensemble = SoftVotingEnsemble(models, device=device)

# Evaluate ensemble
# ensemble_results = ensemble.evaluate(test_loader_weighted)

print("Ensemble evaluation code ready")

---
# 5. SimCLR Contrastive Pretraining

Self-supervised pretraining with contrastive learning.

In [ ]:
class ECGAugmentation:
    """
    Augmentation transforms for 1D ECG signals
    """
    @staticmethod
    def add_noise(x, noise_level=0.05):
        """Add Gaussian noise"""
        noise = torch.randn_like(x) * noise_level
        return x + noise
    
    @staticmethod
    def scale(x, scale_range=(0.8, 1.2)):
        """Random amplitude scaling"""
        scale_factor = torch.FloatTensor(1).uniform_(*scale_range).to(x.device)
        return x * scale_factor
    
    @staticmethod
    def time_shift(x, shift_range=10):
        """Random time shift (circular)"""
        shift = torch.randint(-shift_range, shift_range, (1,)).item()
        return torch.roll(x, shift, dims=-1)
    
    @staticmethod
    def time_mask(x, mask_ratio=0.1):
        """Random time masking"""
        length = x.size(-1)
        mask_length = int(length * mask_ratio)
        start = torch.randint(0, length - mask_length, (1,)).item()
        x_masked = x.clone()
        x_masked[..., start:start+mask_length] = 0
        return x_masked
    
    @staticmethod
    def apply_random_augmentations(x, num_augs=2):
        """Apply random combination of augmentations"""
        augmentations = [
            lambda x: ECGAugmentation.add_noise(x, noise_level=0.03),
            lambda x: ECGAugmentation.scale(x, scale_range=(0.9, 1.1)),
            lambda x: ECGAugmentation.time_shift(x, shift_range=5),
            lambda x: ECGAugmentation.time_mask(x, mask_ratio=0.05)
        ]
        
        selected_augs = random.sample(augmentations, num_augs)
        for aug in selected_augs:
            x = aug(x)
        
        return x

In [ ]:
class ProjectionHead(nn.Module):
    """
    Projection head for contrastive learning
    Maps encoder features to embedding space
    """
    def __init__(self, input_dim, hidden_dim=256, output_dim=128):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.projection(x)

In [ ]:
class NTXentLoss(nn.Module):
    """
    Normalized Temperature-scaled Cross Entropy Loss (NT-Xent)
    Used in SimCLR for contrastive learning
    """
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, z_i, z_j):
        """
        Args:
            z_i: embeddings from augmentation 1 [B, D]
            z_j: embeddings from augmentation 2 [B, D]
        """
        batch_size = z_i.size(0)
        
        # Normalize embeddings
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)
        
        # Concatenate
        z = torch.cat([z_i, z_j], dim=0)  # [2B, D]
        
        # Compute similarity matrix
        sim_matrix = torch.mm(z, z.t()) / self.temperature  # [2B, 2B]
        
        # Create mask to exclude self-similarity
        mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
        sim_matrix = sim_matrix.masked_fill(mask, -9e15)
        
        # Positive pairs: (i, i+B) and (i+B, i)
        pos_sim = torch.cat([
            sim_matrix[range(batch_size), range(batch_size, 2 * batch_size)],
            sim_matrix[range(batch_size, 2 * batch_size), range(batch_size)]
        ])
        
        # Compute loss
        loss = -pos_sim + torch.logsumexp(sim_matrix, dim=1)
        loss = loss.mean()
        
        return loss

In [ ]:
class SimCLR_ECG(nn.Module):
    """
    SimCLR framework for ECG signals
    """
    def __init__(self, encoder, encoder_dim, projection_dim=128):
        super().__init__()
        self.encoder = encoder
        self.projection_head = ProjectionHead(encoder_dim, hidden_dim=256, output_dim=projection_dim)
    
    def forward(self, x):
        # Encode
        h = self.encoder(x)  # [B, encoder_dim]
        # Project
        z = self.projection_head(h)  # [B, projection_dim]
        return h, z

In [ ]:
def pretrain_simclr(encoder, train_loader, epochs=50, temperature=0.5, device='cuda'):
    """
    Pretrain encoder using SimCLR contrastive learning
    
    Args:
        encoder: backbone encoder (e.g., CNN without classification head)
        train_loader: DataLoader with unlabeled ECG data
        epochs: number of pretraining epochs
        temperature: temperature for NT-Xent loss
        device: device to train on
    
    Returns:
        pretrained encoder
    """
    # Create SimCLR model
    # Assume encoder outputs feature vector of size 128 (adjust as needed)
    simclr_model = SimCLR_ECG(encoder, encoder_dim=128, projection_dim=128).to(device)
    
    # Loss and optimizer
    criterion = NTXentLoss(temperature=temperature)
    optimizer = torch.optim.AdamW(simclr_model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    print("="*80)
    print("SimCLR Contrastive Pretraining")
    print("="*80)
    
    for epoch in range(epochs):
        simclr_model.train()
        total_loss = 0.0
        
        for batch_x, _ in train_loader:
            batch_x = batch_x.to(device)
            
            # Generate two augmented views
            x_i = ECGAugmentation.apply_random_augmentations(batch_x, num_augs=2)
            x_j = ECGAugmentation.apply_random_augmentations(batch_x, num_augs=2)
            
            # Forward pass
            _, z_i = simclr_model(x_i)
            _, z_j = simclr_model(x_j)
            
            # Compute contrastive loss
            loss = criterion(z_i, z_j)
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        scheduler.step()
        
        if (epoch + 1) % 5 == 0:
            avg_loss = total_loss / len(train_loader)
            print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")
    
    print("\nPretraining complete!")
    return simclr_model.encoder

In [ ]:
def finetune_classifier(encoder, train_loader, test_loader, num_classes=4, epochs=30, device='cuda'):
    """
    Fine-tune pretrained encoder for classification
    
    Args:
        encoder: pretrained encoder from SimCLR
        train_loader: labeled training data
        test_loader: labeled test data
        num_classes: number of classes
        epochs: fine-tuning epochs
        device: device
    
    Returns:
        fine-tuned model
    """
    # Add classification head
    model = nn.Sequential(
        encoder,
        nn.Linear(128, num_classes)  # Adjust input size based on encoder output
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    print("="*80)
    print("Fine-tuning for Classification")
    print("="*80)
    
    best_f1 = 0.0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        scheduler.step()
        
        # Evaluate
        if (epoch + 1) % 5 == 0:
            model.eval()
            all_preds = []
            all_targets = []
            
            with torch.no_grad():
                for batch_x, batch_y in test_loader:
                    batch_x = batch_x.to(device)
                    outputs = model(batch_x)
                    preds = torch.argmax(outputs, dim=1).cpu().numpy()
                    all_preds.extend(preds)
                    all_targets.extend(batch_y.numpy())
            
            acc = accuracy_score(all_targets, all_preds)
            f1 = f1_score(all_targets, all_preds, average='macro')
            
            print(f"Epoch [{epoch+1}/{epochs}] Loss: {train_loss/len(train_loader):.4f} "
                  f"Acc: {acc:.4f} F1: {f1:.4f}")
            
            if f1 > best_f1:
                best_f1 = f1
                torch.save(model.state_dict(), 'best_simclr_finetuned.pth')
    
    print(f"\nBest F1: {best_f1:.4f}")
    return model

In [ ]:
# Example: Pretrain and fine-tune
# Create encoder (without classification head)
# class SimpleEncoder(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.conv_layers = nn.Sequential(
#             nn.Conv1d(1, 32, 7, padding=3),
#             nn.BatchNorm1d(32),
#             nn.ReLU(),
#             nn.Conv1d(32, 64, 5, padding=2),
#             nn.BatchNorm1d(64),
#             nn.ReLU(),
#             nn.AdaptiveAvgPool1d(1)
#         )
#         self.fc = nn.Linear(64, 128)
#     
#     def forward(self, x):
#         x = self.conv_layers(x).squeeze(-1)
#         x = self.fc(x)
#         return x

# encoder = SimpleEncoder().to(device)

# Pretrain
# pretrained_encoder = pretrain_simclr(encoder, train_loader_weighted, epochs=50, temperature=0.5, device=device)

# Fine-tune
# finetuned_model = finetune_classifier(pretrained_encoder, train_loader_weighted, test_loader_weighted, 
#                                       num_classes=4, epochs=30, device=device)

print("SimCLR pretraining code ready")

---
# 6. Grad-CAM Visualization

Visualize which parts of ECG signal are important for classification.

In [ ]:
class GradCAM:
    """
    Gradient-weighted Class Activation Mapping for 1D CNN
    Visualizes which parts of ECG signal influence classification
    """
    def __init__(self, model, target_layer):
        """
        Args:
            model: PyTorch model
            target_layer: layer to compute gradients (e.g., last conv layer)
        """
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        """Hook to save forward activations"""
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        """Hook to save backward gradients"""
        self.gradients = grad_output[0].detach()
    
    def generate_cam(self, input_signal, target_class=None):
        """
        Generate class activation map
        
        Args:
            input_signal: input tensor [1, C, L]
            target_class: class to visualize (None = predicted class)
        
        Returns:
            cam: class activation map [L]
            pred_class: predicted class
        """
        self.model.eval()
        input_signal = input_signal.requires_grad_()
        
        # Forward pass
        output = self.model(input_signal)
        pred_class = torch.argmax(output, dim=1).item()
        
        # Use target class or predicted class
        if target_class is None:
            target_class = pred_class
        
        # Backward pass
        self.model.zero_grad()
        class_score = output[0, target_class]
        class_score.backward()
        
        # Get gradients and activations
        gradients = self.gradients[0]  # [C, L]
        activations = self.activations[0]  # [C, L]
        
        # Global average pooling of gradients
        weights = torch.mean(gradients, dim=1, keepdim=True)  # [C, 1]
        
        # Weighted combination of activation maps
        cam = torch.sum(weights * activations, dim=0)  # [L]
        
        # ReLU and normalize
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        
        return cam.cpu().numpy(), pred_class
    
    def visualize(self, input_signal, target_class=None, save_path=None):
        """
        Visualize Grad-CAM overlay on ECG signal
        
        Args:
            input_signal: input tensor [1, 1, L]
            target_class: class to visualize
            save_path: path to save figure
        """
        cam, pred_class = self.generate_cam(input_signal, target_class)
        
        # Get original signal
        signal = input_signal[0, 0].cpu().numpy()
        
        # Interpolate CAM to match signal length
        from scipy.ndimage import zoom
        if len(cam) != len(signal):
            cam = zoom(cam, len(signal) / len(cam))
        
        # Create figure
        fig, axes = plt.subplots(3, 1, figsize=(14, 8))
        
        # Plot 1: Original ECG signal
        axes[0].plot(signal, color='black', linewidth=1.5)
        axes[0].set_title('Original ECG Signal', fontsize=12, fontweight='bold')
        axes[0].set_ylabel('Amplitude')
        axes[0].grid(True, alpha=0.3)
        
        # Plot 2: Grad-CAM heatmap
        im = axes[1].imshow(cam.reshape(1, -1), cmap='jet', aspect='auto', interpolation='bilinear')
        axes[1].set_title('Grad-CAM Activation Map', fontsize=12, fontweight='bold')
        axes[1].set_ylabel('Importance')
        axes[1].set_yticks([])
        plt.colorbar(im, ax=axes[1], orientation='vertical', label='Activation')
        
        # Plot 3: Overlay
        axes[2].plot(signal, color='black', linewidth=1.5, label='ECG Signal')
        
        # Color signal by importance
        for i in range(len(signal) - 1):
            color_intensity = cam[i]
            color = plt.cm.Reds(color_intensity)
            axes[2].plot([i, i+1], [signal[i], signal[i+1]], 
                        color=color, linewidth=2, alpha=0.7)
        
        axes[2].set_title(f'Grad-CAM Overlay (Predicted: {int_to_class[pred_class]})', 
                         fontsize=12, fontweight='bold')
        axes[2].set_xlabel('Time (samples)')
        axes[2].set_ylabel('Amplitude')
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Saved visualization to {save_path}")
        
        plt.show()
        
        return cam, pred_class

In [ ]:
# Example: Apply Grad-CAM to a sample
# Assuming you have a trained CNN model

# model = SimpleCNN(num_classes=4).to(device)
# model.load_state_dict(torch.load('best_model.pth'))

# Get target layer (last conv layer)
# For SimpleCNN, it's conv3
# target_layer = model.conv3

# Create Grad-CAM
# grad_cam = GradCAM(model, target_layer)

# Get a sample ECG
# sample_idx = 100
# sample_signal = torch.FloatTensor(X_test[sample_idx]).unsqueeze(0).unsqueeze(0).to(device)
# true_label = y_test[sample_idx]

# Visualize
# cam, pred_class = grad_cam.visualize(sample_signal, target_class=None, save_path='gradcam_ecg.png')

# print(f"True label: {int_to_class[true_label]}")
# print(f"Predicted: {int_to_class[pred_class]}")

print("Grad-CAM visualization code ready")

---
# Summary

This notebook implements 6 advanced techniques for ECG classification:

1. **Patient-wise Cross Validation**: Prevents data leakage using StratifiedGroupKFold
2. **Focal Loss + WeightedRandomSampler + MixUp**: Better class imbalance handling without SMOTE
3. **Temporal Convolutional Network (TCN)**: Dilated causal convolutions for time-series
4. **Soft Voting Ensemble**: Combines multiple models for better performance
5. **SimCLR Contrastive Pretraining**: Self-supervised learning for feature extraction
6. **Grad-CAM Visualization**: Interpretable model decisions

All implementations are modular and can be adapted to your specific needs.